---
###  Validation of the microstate model: autocorrelation test (RMSD)

Lets call $\theta (\vec{x})$ an observable of the molecular trajectory.
- Construct the vector of observables $\mathbf{\theta} = (\theta_1, \, \theta_N)$ containing the value of the observable $\theta$ in each markov microstate.


    0. For each cluster center, I take the state closest to the cluster center as the representative state of the markov microstate;
    1. For each representative state, extract its temporal frame index and use it to retrieve the corresponding value of the RMSD from the crystal structure.

- Compute the model predicted autocorrelation according to:

$$
ACF_\theta(k) = \frac{\sum_{i=1}^{N}\, \lambda_i^k\, \langle\mathbf{\theta}, \mathbf{r_i} \rangle^2}{\sum_{i=1}^{N}\, \langle\mathbf{\theta}, \mathbf{r_i}\rangle^2}
$$

where the $\mathbf{r}_i$ are the right eigenvectors of the transition matrix.

- Compare with the empirical autocorrelation.

In [ ]:
from scipy.signal import correlate
from deeptime.markov.tools.analysis import correlation


def get_empirical_autocorrelation(obs):
    obs = obs - obs.mean()
    acf = correlate(obs, obs, mode="full")
    acf = acf[acf.size // 2:]      # keep non-negative lags
    acf /= acf[0]                  # normalize so acf[0] = 1
    return acf


def get_msm_autocorrelation(obs, msm, times_array):
    """
    obs: array of observables, one for each microstate
    msm: object of the class deeptime::MarkovStateModelCollection
    """
    # center the observable using the stationary distribution
    obs_mean = np.dot(msm.stationary_distribution, obs)
    obs = obs - obs_mean
    variance = correlation(T = msm.transition_matrix, 
                                obs1 = obs, 
                                obs2=None, 
                                times=(0,), 
                                k=None, 
                                ncv=None)[0]
    
    times_array_in_units_of_tau = np.array(times_array / msm.lagtime, dtype = int)
    acf = correlation(T = msm.transition_matrix, 
                                obs1 = obs, 
                                obs2=None, 
                                times=times_array_in_units_of_tau, 
                                k=None, 
                                ncv=None)
    
    return acf/variance


crystal_dist = np.loadtxt(
    data_folder + "hp35.crystaldists",
    delimiter=" ",
    dtype=float
)


crystal_dist = np.loadtxt(
    data_folder + "hp35.crystaldists",
    delimiter=" ",
    dtype=float
)
RMSD_timeseries = np.sqrt(np.mean(crystal_dist**2, axis=1))  # [nm]

theta_dict = {} # one array of observables per each msm model

for i, name in enumerate(trajectories_labels):
    traj = micro_trajectory_dict[name]
    msm = msm_dict[name]
    theta_array = np.ones(msm.n_states)  * (-1)
    for j in np.arange(0, msm.n_states):
        representative_frames = np.where(traj == j)[0]
        RMSD_values = RMSD_timeseries[representative_frames]
        theta_array[j] = np.mean(RMSD_values)
    theta_dict[name] = theta_array


empirical_acf = get_empirical_autocorrelation(RMSD_timeseries)



max_time = 100000 # in frames, corresponds to 20 microseconds
print("Max time for ACF (frames)", max_time)
print("Max time for ACF (nanoseconds)", max_time * time_step_nanos) # 20 microseconds
model_acf_dict = {}
for name in trajectories_labels:
    theta_array = theta_dict[name]
    model_acf_dict[name] = get_msm_autocorrelation(theta_dict[name], msm_dict[name], times_array=np.arange(0, max_time)) 

In [ ]:
fig, ax = plt.subplots(figsize = (5,4))


ax.plot(np.arange(len(empirical_acf)), empirical_acf, label = "empirical")
for name in trajectories_labels:
    model_acf = model_acf_dict[name]
    ax.plot(np.arange(len(model_acf)), model_acf, label = f"{name}-model")

ax.set_xscale("log", base=10)
# ticks at powers of 10 frames
xticks = [1, 10, 100, 1000, 10000, 100000]
xticks = [x for x in xticks if x < len(empirical_acf)]
ax.set_xticks(xticks)
ax.set_xticklabels([f"{x*time_step_micros:.1f}" for x in xticks])

ax.legend()
ax.set_ylabel("ACF RMSD")
ax.set_xlabel(f"Time [$\mu$s]")
ax.grid()
plt.ylim(-0.2, 1.2)
plt.show()